# What the sloppy server does for an agent

This notebook reads the run records under `results/` (or the folder in `BENCH_RESULTS`) and the
grades beside them, and draws every condition beside every other. README.md § "The protocol" is
what the numbers mean; § "What counts as better" is the rule the verdict table applies.

Every figure shows every condition that ran. **Nothing is averaged across subjects or agents**: every
per-condition figure is drawn once per subject and agent (the panel's title names both), and the
verdict table compares conditions within one subject, agent and task. Two subjects' runs in one
results folder are two sets of panels, never one mean. A group of fewer than two runs is drawn without an
interval and labelled `n=1`; a figure with no runs at all says so instead of drawing.

## Findings so far — 2026-10-29, a first look with one repetition per cell

Backend: the Claude Code installed on the developer's machine (2.1.311), model `claude-sonnet-5-5`, subject this repository at `f420d155`, eight tasks × six conditions, seed 2; $2.99 for the runs and $0.78 for the judge. **By the protocol's own rule every difference below is a tie**: one repetition is under the five-rep bar. This is what one batch shows, to be tested with more.

- **Scores.** Every condition that had the documents scored 0.96–0.98 on the blind judge and 1.00 on every objective check. `nodocs` scored 0.73: it failed the why-questions and the brief (0.02), whose answers are not in the code. The documents matter; how they reached the agent mattered far less for correctness.
- **Cost per task, mean.** pointed $0.040, bare $0.049, mcp $0.062, mcp+pointed $0.065, dump $0.068, nodocs $0.090. **Time to answer.** dump 13 s, pointed 14 s, bare 15 s, mcp 19 s, mcp+pointed 22 s, nodocs 29 s.
- **Where the notes cost more for the same score:** the three why-questions and the brief. Claude Code reads `CLAUDE.md` (a symlink to AI.md) in the full variant, so `bare` answered a why-question in one turn for about $0.01 and 7 s, while `mcp` searched and read three to six notes for $0.03–0.06 and 15–18 s. The notes held nothing the agent needed that AI.md had not already given it.
- **Where the notes cost less for the same score:** the drift check ($0.037 against $0.067 bare) and locating the proxy exception ($0.033 against $0.044): one `search_notes` led straight to the anchored code.
- The notes were not used at all for the code change, and `mcp+pointed` never beat `mcp`.
- `dump` matched the best scores at 440k input tokens per task on average; with Sonnet's prompt cache that cost only 1.4× bare, so it is the condition most sensitive to pricing and to cache behaviour.
- **Limits of this batch.** `bare`, `pointed` and `mcp` all had AI.md through `CLAUDE.md`, so `nodocs` is the only true absence of documents. The objective graders are coarse regex checks and saturated at 1.00. The first batch (seed 1) is kept under `results/superseded/` and drawn nowhere, because the change task's wording was ambiguous and four agents read it the other way.
- **What this points at building, to be measured as `--config` variants:** a `read_note` that answers only the sections asked for and a `search_notes` that answers titles and refs first, so a why-question costs one call rather than four; a neighbourhood read of n hops; and a line in AGENT.md telling an agent to use the notes for sources and for what the documents do not say, not for what it already holds.

## Second read — 2026-10-29, the interleaved batch (seed 3)

Ninety-six runs: two server builds interleaved in one seeded order, sharing the hour and the cache, under the neutral mcp wording (§ Revisions). Still one repetition per cell, so still ties by the protocol. What it shows:

- **Reported cost is not a stable metric on Claude Code.** `bare` cost $0.037 under one config and $0.099 under the other in the same batch, on trees that differ by nothing the agent reads: the difference is prompt-cache hits, which depend on order and on the folder's path. Compare input-token counts by class, which the figures draw, and treat the dollar figure as what it is, the bill for that run.
- **A confound, stated in `results/README.md`:** the `default` server binary was the developer's `mcp/brief` build (it carries a `brief` act), not the f420d155 code the variant tree is at. So this batch compares the `brief` build against the `lean` build, both on the same tasks.
- **Under the neutral wording, the `brief` build's agent left the notes alone on the three why-questions**: 27k input tokens, the same as `bare`, the same score, one `brief` call on the drift and locate tasks. **The `lean` build's agent still searched and read two to four whole notes per why-question**, 100–125k tokens, for the same score. The lean defaults cut the listing but did not change the agent's choice to read whole notes, and it never asked for only the `Why` section on its own. What moved the overhead was the wording in front of the agent and an act shaped for the question, not smaller defaults.
- Scores did not move anywhere: 0.96–0.98 with documents, about 0.7 without.
- **Next:** rebuild the f420d155 server so `default` means what it says, merge the `brief` act and the lean acts, and run default, lean and brief as three configs in one batch with five repetitions before calling any of this a result.

## Third read — 2026-10-30, the first complete batch (seed 5, two repetitions)

Two hundred and twenty-four runs: eight tasks, seven conditions including `notes-only`, the f420d155 server (`default`) and the merged build (`merged`: lean acts, answer-first wording, the `brief` act) interleaved in one order; every run finished and was judged, $19.82. Two repetitions per cell is still under the protocol's five-rep bar, so these are consistent readings, not results.

- **The notes alone carry the knowledge.** With every document removed and only the container left, `notes-only` scored 0.91–0.92 against 0.73–0.75 for `nodocs`, at fewer input tokens (72–85k against 89–96k) and less time (19–22 s against 26–32 s). On the why-questions it went from 0.43–0.88 to 0.96–0.99. This is the cell the earlier batches did not have, and it is the claim the product makes.
- **The merged build removed the overhead where the documents already answer.** On the `mcp` condition it used 74k input tokens against the f420d155 server's 103k, 12 server calls against 32, 15 s against 18 s, for the same score (0.95 against 0.96). On the three why-questions its agent used 28k tokens, the same as `bare`, where the old server read 113–123k. One cell went the other way: `locate-move-rule` under `mcp` scored 0.81 on the merged build against 0.98, from one poor run of two.
- **The massive prompt is the worst documented condition on this repo.** `dump` cost three times `bare`, scored 0.92–0.93, and failed the file-locating task (0.62 and 0.76) that every other documented condition passed: too much context, not too little.
- **`pointed` and `bare` remain cheapest and best on this subject**, because its prose documents fit a window and are unusually complete. The brief (`brief-protocol`) stays hard for `notes-only` (0.46–0.47 at 118–164k tokens): a map of the graph served first is the knob this points at.
- **Reported cost is now comparable within the batch** because both builds shared one order; across batches it still is not.

## Fourth read — 2026-10-30, the syr subject (seed 6, two repetitions)

One hundred and twenty-six runs on a second subject: the syr repository, whose documentation is a 47-page site (62k words) carried into 61 notes that morning, and whose `CLAUDE.md` is 500 words. Nine tasks, seven conditions, the merged build only, every run ok and judged, $11.53.

- **The notes alone equal the documents.** `notes-only` scored 0.92, the same as `bare` (0.92), `pointed` (0.93) and `dump` (0.93), and far above `nodocs` (0.72). On this subject an agent under `bare` has almost nothing in context and finds the site's pages itself; the notes give it the same answers at 86k input tokens against 60k, in the same time. On the questions the documents settle (`why-mandates-built` 0.47 → 0.96; `why-root-on-devices` 0.58 → 0.91; the drift check 0.65 → 0.97) the notes carried everything the pages did.
- **The server on top of the documents still adds cost without score**: `mcp` 91k tokens and 0.92 against `bare` 60k and 0.92. The agent read two to four notes per question (25 `read_note`, 13 `search_notes`, 6 `brief` across the condition) where grep over the pages cost less. On a subject whose documents are in the repository and fit a window, the notes compete with the pages rather than add to them.
- **The brief is the hardest task everywhere** (0.31 to 0.82) and the one where `mcp` lost most (0.65 at 114k tokens): an agent has no map of the graph to brief from. The same knob as on the first subject.
- **`dump` works here** (0.93, fastest at 14 s) at 443k tokens and 2.3× the cost: the site fits a window, so the massive prompt is a price, not a failure, on this subject.
- Still two repetitions per cell; the preregistered rule reads all of this as ties. What is consistent across both subjects: notes beat nothing by a wide margin, notes match documents, and the server only pays for itself where the documents are not already in front of the agent.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import figures

RESULTS = Path(os.environ.get("BENCH_RESULTS", "results"))
MIN_REPS = 5
BOOTSTRAP = 2000
SEED = 0

PALETTE = figures.PALETTE
CONDITION_ORDER = figures.CONDITION_ORDER
TOKEN_CLASSES = ["input", "cache_write", "cache_read", "output", "reasoning"]
figures.configure()


def read_jsonl(folder):
    rows = []
    for file in sorted(folder.glob("*.jsonl")):
        for line in file.read_text().splitlines():
            if line.strip():
                row = json.loads(line)
                row["_file"] = file.name
                rows.append(row)
    return rows


records = read_jsonl(RESULTS)
grades = read_jsonl(RESULTS / "grades")
print(f"{len(records)} run records and {len(grades)} grade records in {RESULTS.resolve()}")

In [ ]:
def latest(run_id, grader):
    found = [g for g in grades if g["run_id"] == run_id and g["grader"] == grader]
    return found[-1]["score"] if found else None


def score_of(r):
    """lib/report.mjs scoreOf: a started run that did not finish ok scores 0."""
    if r["status"] != "ok":
        return 0.0
    run_grade = r.get("grade") or {}
    parts = []
    for grader in ("objective", "judge"):
        s = latest(r["run_id"], grader)
        if s is None:
            s = run_grade.get(grader)
        if s is not None:
            parts.append(s)
    if parts:
        return float(np.mean(parts))
    return run_grade.get("score", np.nan)


def flatten(r):
    tokens = r.get("tokens", {})
    unknown = set(r.get("tokens_unknown", []))
    cost = r.get("cost_usd", {})
    timing = r.get("timing", {})
    stage_a = r.get("stage_a")
    row = {
        "run_id": r["run_id"],
        "file": r["_file"],
        "timestamp": pd.to_datetime(r["timestamp"]),
        "subject": r["subject"].get("name", "sloppy"),
        "sha": r["subject"]["sha"][:12],
        "variant": r["subject"]["variant"],
        "backend": r["backend"]["kind"],
        "model": r["backend"]["model"],
        "provider": r["backend"].get("provider"),
        "version": r["backend"].get("version"),
        "agent": f'{r["backend"]["kind"]} / {r["backend"]["model"]}',
        "condition": r["condition"],
        "config": r.get("config") or "shipped",
        "pack": r["task"]["pack"],
        "task": r["task"]["id"],
        "family": r["task"]["family"],
        "rep": r["rep"],
        "seed": r["seed"],
        "attempt": r.get("attempt", 0),
        "status": r["status"],
        "started": r["status"] != "budget",
        "wall_s": (timing.get("wall_ms", np.nan) + (stage_a["timing"]["wall_ms"] if stage_a else 0)) / 1000,
        "first_tool_s": timing.get("first_tool_ms", np.nan) / 1000,
        "answer_s": timing.get("answer_ms", np.nan) / 1000,
        "cost_usd": figures.run_cost(r),
        "turns": r.get("turns", 0),
        "tool_calls": sum(r.get("tool_calls", {}).values()),
        "mcp_calls": sum(r.get("mcp_calls", {}).values()),
        "score": score_of(r),
    }
    for c in TOKEN_CLASSES:
        first = (stage_a or {}).get("tokens", {}).get(c, 0)
        told = c not in unknown and c not in set((stage_a or {}).get("tokens_unknown", []))
        row[f"tok_{c}"] = tokens.get(c, 0) + first if told else np.nan
    return row


runs = pd.DataFrame([flatten(r) for r in records])
mcp_acts = pd.DataFrame(
    [{"subject": r["subject"].get("name", "sloppy"), "condition": r["condition"], "agent": f'{r["backend"]["kind"]} / {r["backend"]["model"]}', "act": act, "calls": n}
     for r in records for act, n in r.get("mcp_calls", {}).items()]
)
started = runs[runs["started"]] if len(runs) else runs
conditions = [c for c in CONDITION_ORDER if c in set(runs.get("condition", []))] + sorted(
    set(runs.get("condition", [])) - set(CONDITION_ORDER)
)
colour = {c: figures.style(c)[0] for c in conditions}
marker = {c: figures.style(c)[1] for c in conditions}
curve_runs = figures.runs_frame(records, grades)


def bootstrap_ci(values, stat=np.mean):
    values = np.asarray([v for v in values if not pd.isna(v)], dtype=float)
    if len(values) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(SEED)
    draws = [stat(rng.choice(values, len(values), replace=True)) for _ in range(BOOTSTRAP)]
    return tuple(np.percentile(draws, [2.5, 97.5]))


def summary(frame, by, column):
    rows = []
    for key, g in frame.groupby(by, sort=False):
        values = g[column].dropna()
        lo, hi = bootstrap_ci(values)
        rows.append({**dict(zip(by, key if isinstance(key, tuple) else (key,))),
                     "n": len(values), "mean": values.mean() if len(values) else np.nan, "lo": lo, "hi": hi})
    return pd.DataFrame(rows)


def panels(frame):
    """(label, rows) per subject and agent, so no figure averages across either."""
    if not len(frame):
        return []
    return [(f"{subject} · {agent}", g) for (subject, agent), g in frame.groupby(["subject", "agent"], sort=True)]


def panel_axes(frame, ncols=1, width=7, height=3.5):
    found = panels(frame)
    fig, axes = plt.subplots(len(found), ncols, figsize=(width, height * len(found)), squeeze=False)
    return fig, [(row, label, g) for row, (label, g) in zip(axes, found)]


def ran_in(frame):
    return [c for c in conditions if c in set(frame["condition"])]


def nothing_to_draw(what):
    print(f"No runs to draw {what} from.")

## Provenance

What produced these numbers: the subject commit, the agent and model, the configuration, when, and
from which file. `results/README.md` holds who ran each batch and on what machine. This table
cannot tell whether two batches ran on the same machine.

In [ ]:
if len(runs):
    provenance = runs.groupby(["file", "subject", "sha", "backend", "model", "provider", "version", "config"], dropna=False).agg(
        runs=("run_id", "count"), seeds=("seed", lambda s: ", ".join(map(str, sorted(set(s))))),
        first=("timestamp", "min"), last=("timestamp", "max"),
    ).reset_index()
    display(provenance)
else:
    nothing_to_draw("a provenance table")

## One run, by its id

`show_run(run_id)` opens one run the way `node bench.mjs report --run <run_id>` does: what it was sent (kept on the record, or rebuilt from the commits that made it for a run before prompts were kept), a timeline of its model calls and tool calls, its answer and its grades. A run made before transcripts were kept shows the calls its record counted, without their times. The cell below opens the most recent run.

In [ ]:
import gzip


def sent_of(r):
    if r.get("sent"):
        return {**r["sent"], "reconstructed": False}
    kept = RESULTS / "prompts" / f'{r["run_id"]}.json'
    if not kept.exists():
        return None
    k = json.loads(kept.read_text())
    sent = dict(k["sent"])
    sent["system"] = gzip.decompress((RESULTS / "prompts" / sent.pop("system_file")).read_bytes()).decode()
    return {**sent, "reconstructed": True, "from": k["from"]}


def transcript_of(run_id):
    file = RESULTS / "transcripts" / f"{run_id}.jsonl.gz"
    if not file.exists():
        return None
    return [json.loads(line) for line in gzip.decompress(file.read_bytes()).decode().splitlines() if line.strip()]


def transcript_calls(events):
    """lib/run.mjs transcriptCalls: each tool call once, at the time it first arrived."""
    seen = {}

    def visit(node, ms):
        if isinstance(node, list):
            for n in node:
                visit(n, ms)
        elif isinstance(node, dict):
            name = node.get("name") if node.get("type") == "tool_use" else (node.get("function") or {}).get("name") if node.get("type") == "function" else None
            if name and node.get("id") and not (node["id"] in seen and seen[node["id"]][0] <= ms):
                seen[node["id"]] = (ms, name)
            for v in node.values():
                visit(v, ms)

    for e in events:
        visit(e["event"], e["ms"])
    return sorted(seen.values())


def show_run(run_id, system_chars=1200):
    found = [r for r in records if r["run_id"] == run_id]
    if not found:
        print(f"No run in {RESULTS} has the id {run_id}.")
        return
    r = found[0]
    sent, events = sent_of(r), transcript_of(run_id)
    print(f'{run_id}  {r["subject"].get("name", "sloppy")} at {r["subject"]["sha"][:12]}  {r["condition"]}  {r["task"]["pack"]}/{r["task"]["id"]}  rep {r["rep"]}  {r["status"]}')
    if sent is None:
        print("\nWhat this run was sent was not kept.")
    else:
        where = f' (rebuilt from the bench at {sent["from"]["bench_commit"][:12]})' if sent["reconstructed"] else ""
        print(f'\nSent{where}: model {sent["model"]}, tools {", ".join(sent["tools"]) or "none"}, sloppy tools {"offered" if sent["mcp"] else "not offered"}')
        system = sent["system"]
        cut = system if len(system) <= system_chars else f"{system[:system_chars]}\n... {len(system) - system_chars} more characters"
        print(f"\nSystem prompt ({len(system)} characters):\n{cut}\n\nTask:\n{sent['prompt']}")
    moments = [(it["at_ms"], f'model call {i + 1}: {it["input"] + it["cache_write"] + it["cache_read"]} tokens in, {it["output"]} out') for i, it in enumerate(r.get("iterations", []))]
    moments += [(ms, f"tool call: {name}") for ms, name in transcript_calls(events or [])]
    timing = r.get("timing", {})
    if events is None and "first_tool_ms" in timing:
        moments.append((timing["first_tool_ms"], "first tool call"))
    if "answer_ms" in timing:
        moments.append((timing["answer_ms"], "answer began"))
    moments.append((timing.get("wall_ms", 0), f'run ended: {r["status"]}'))
    print("\nTimeline:")
    for ms, what in sorted(moments, key=lambda m: m[0]):
        print(f"  {ms / 1000:7.1f}s  {what}")
    counts = lambda calls: ", ".join(f"{k} {v}" for k, v in calls.items()) or "none"
    print(f'\nTool calls: {counts(r.get("tool_calls", {}))}\nSloppy calls: {counts(r.get("mcp_calls", {}))}')
    print(f"Transcript: {len(events)} events" if events is not None else "Transcript: not kept for this run")
    print(f'\nAnswer:\n{(r.get("output") or {}).get("text") or "(none)"}')
    print("\nGrades:")
    for g in [g for g in grades if g["run_id"] == run_id]:
        print(f'  {g["grader"]}: {g["score"]}')


if records:
    show_run(max(records, key=lambda r: r["timestamp"])["run_id"])
else:
    print("No runs to open.")


## Coverage

How many runs each condition got on each task, and how they ended: `ok`, `error`, `timeout`, or
`budget` (never started, because the batch's budget was spent), per subject and agent. A condition
with more failures has fewer runs behind its means; a failed run still counts, at score 0, in what
it spent. This does not show why a run failed; the record's `error` does.

In [ ]:
if len(runs):
    for label, g in panels(runs):
        print(label)
        display(g.pivot_table(index="task", columns="condition", values="run_id", aggfunc="count", fill_value=0)
                .reindex(columns=ran_in(g)))
    fig, rows = panel_axes(runs)
    status_colour = {"ok": PALETTE[2], "error": PALETTE[5], "timeout": PALETTE[1], "budget": "#999999"}
    for (ax,), label, g in rows:
        status = g.pivot_table(index="condition", columns="status", values="run_id", aggfunc="count", fill_value=0).reindex(ran_in(g))
        bottom = np.zeros(len(status))
        for s in [c for c in ["ok", "error", "timeout", "budget"] if c in status.columns]:
            ax.bar(status.index, status[s], bottom=bottom, label=s, color=status_colour[s])
            bottom += status[s].to_numpy()
        ax.set_ylabel("runs")
        ax.set_title(f"How each condition's runs ended: {label}")
        ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
else:
    nothing_to_draw("coverage")

## Cost by condition and task

Mean USD per run, by condition, on each task, with a 95% bootstrap interval, one panel per subject,
agent and model. Cost is the one computed from list prices where there is one, else the one the agent
reported. A failed run is in the mean at what it spent. It does not show what the cost bought;
the score figures do.

In [ ]:
def grouped_bars(frame, column, ylabel, title):
    if not len(frame) or frame[column].dropna().empty:
        nothing_to_draw(title.lower())
        return
    fig, rows = panel_axes(frame, width=10, height=3.6)
    for (ax,), label, g in rows:
        s = summary(g, ["task", "condition"], column)
        tasks = list(dict.fromkeys(s["task"]))
        width = 0.8 / max(1, len(conditions))
        for i, c in enumerate(conditions):
            part = s[s["condition"] == c].set_index("task").reindex(tasks)
            x = np.arange(len(tasks)) + i * width
            err = np.vstack([part["mean"] - part["lo"], part["hi"] - part["mean"]])
            ax.bar(x, part["mean"], width, color=colour[c], label=c,
                   yerr=np.where(np.isnan(err), 0, err), capsize=2)
            for xi, n in zip(x, part["n"]):
                if n == 1:
                    ax.text(xi, 0, "n=1", rotation=90, ha="center", va="bottom", fontsize=7)
        ax.set_xticks(np.arange(len(tasks)) + 0.4 - width / 2, tasks, rotation=20, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}: {label}")
    rows[0][0][0].legend(frameon=False, ncol=min(6, len(conditions)), fontsize=8)
    fig.tight_layout()
    plt.show()


grouped_bars(started, "cost_usd", "USD per run", "Cost")

## Tokens by class

Mean tokens per run in each class, stacked, by condition, one panel per
subject and agent. Cache reads are cheap and
cache writes dear, so the stack's height is not its price. A run whose agent could not tell a
class is left out of that class's mean rather than counted as zero. An installed agent's own
system prompt is inside its input classes and is not shown apart.

In [ ]:
if len(started):
    fig, rows = panel_axes(started, width=7, height=4)
    for (ax,), label, g in rows:
        m = g.groupby("condition")[[f"tok_{c}" for c in TOKEN_CLASSES]].mean().reindex(ran_in(g))
        bottom = np.zeros(len(m))
        for i, c in enumerate(TOKEN_CLASSES):
            values = m[f"tok_{c}"].fillna(0).to_numpy()
            ax.bar(m.index, values, bottom=bottom, color=PALETTE[i], label=c)
            bottom += values
        ax.set_title(label)
        ax.set_ylabel("tokens per run")
        ax.tick_params(axis="x", rotation=20)
    rows[0][0][0].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    nothing_to_draw("tokens")

## Turns and tool calls

Mean model turns, tool calls and sloppy calls per run, by condition, one panel per subject and agent; then the sloppy acts called,
in total, by condition, per subject. More calls is not worse in itself; it is what the cost and time figures
pay for. It does not show which calls were useful.

In [ ]:
if len(started):
    fig, rows = panel_axes(started, width=8)
    for (ax,), label, g in rows:
        m = g.groupby("condition")[["turns", "tool_calls", "mcp_calls"]].mean().reindex(ran_in(g))
        x = np.arange(len(m))
        for i, col in enumerate(["turns", "tool_calls", "mcp_calls"]):
            ax.bar(x + i * 0.27, m[col], 0.27, color=PALETTE[i], label=col.replace("_", " "))
        ax.set_xticks(x + 0.27, m.index)
        ax.set_ylabel("per run")
        ax.set_title(f"Turns and calls by condition: {label}")
    rows[0][0][0].legend(frameon=False)
    fig.tight_layout()
    plt.show()
    if len(mcp_acts):
        display(mcp_acts.pivot_table(index=["subject", "act"], columns="condition", values="calls", aggfunc="sum", fill_value=0))
    else:
        print("No sloppy calls were made.")
else:
    nothing_to_draw("turns and calls")

## Time to first tool and to the answer

Seconds from the start of a run to its first tool call, and to the start of its final answer,
by condition, one row of panels per subject and agent, where the agent could tell. A backend that cannot tell first-tool or answer time
leaves those runs out, so a box may stand on fewer runs than the condition had.

In [ ]:
if len(started) and started[["first_tool_s", "answer_s"]].notna().any().any():
    fig, rows = panel_axes(started, ncols=2, width=11, height=3.8)
    for axes, label, g in rows:
        here = ran_in(g)
        for ax, when in zip(axes, ["first_tool_s", "answer_s"]):
            data = [g[g["condition"] == c][when].dropna().to_numpy() for c in here]
            box = ax.boxplot([d if len(d) else [np.nan] for d in data], tick_labels=here, patch_artist=True)
            for patch, c in zip(box["boxes"], here):
                patch.set_facecolor(colour[c])
            for i, d in enumerate(data, 1):
                if len(d) < 2:
                    ax.text(i, ax.get_ylim()[0], f"n={len(d)}", ha="center", va="bottom", fontsize=7)
            ax.set_title(f"{'to first tool' if when == 'first_tool_s' else 'to answer'}: {label}")
            ax.set_ylabel("seconds")
            ax.tick_params(axis="x", rotation=20)
    fig.tight_layout()
    plt.show()
else:
    nothing_to_draw("first-tool and answer times")

### The delivery timeline

The mean run per condition, start to end, one panel per subject and agent: the bar is wall time, the runner's own clock around the
whole run, process start included; the tick is the first tool call and the diamond the start of
the answer. Means hide spread; the boxes above show it.

In [ ]:
if len(started):
    found = panels(started)
    fig, axes = plt.subplots(len(found), 1, figsize=(8, sum(0.8 + 0.5 * g["condition"].nunique() for _, g in found)), squeeze=False)
    for ax, (label, g) in zip(axes[:, 0], found):
        t = g.groupby("condition")[["first_tool_s", "answer_s", "wall_s"]].mean().reindex(ran_in(g))
        for y, (c, row) in enumerate(t.iterrows()):
            ax.barh(y, row["wall_s"], color=colour[c], alpha=0.35)
            if not np.isnan(row["first_tool_s"]):
                ax.plot(row["first_tool_s"], y, "|", color="black", markersize=14)
            if not np.isnan(row["answer_s"]):
                ax.plot(row["answer_s"], y, marker="D", color="black", markersize=5)
        ax.set_yticks(range(len(t)), t.index)
        ax.set_title(f"The mean run, by condition: {label}")
    axes[-1, 0].set_xlabel("seconds from start (bar: wall; |: first tool; ◆: answer)")
    fig.tight_layout()
    plt.show()
else:
    nothing_to_draw("a delivery timeline")

## Score by condition

Mean score per run by condition and task, with a 95% bootstrap interval, one panel per subject,
agent and model. The score is the mean of the objective grader's and the judge's, whichever a run has; a
run that started and did not finish `ok` scores 0. It does not show what the score cost.

In [ ]:
grouped_bars(started, "score", "score (0 to 1)", "Score")

## Quality against cost

Each point is one run: its cost against its score, the condition shown by colour and marker, one panel per subject
and agent.
Up and to the left is better. A cloud that is higher and further right is a trade, not a win.
Runs with no cost that could be told are not drawn.

In [ ]:
drawn = started.dropna(subset=["cost_usd", "score"]) if len(started) else started
if len(drawn):
    fig, rows = panel_axes(drawn, width=7, height=4.5)
    for (ax,), label, g in rows:
        for c in ran_in(g):
            part = g[g["condition"] == c]
            ax.scatter(part["cost_usd"], part["score"], color=colour[c], marker=marker[c], label=f"{c} (n={len(part)})", alpha=0.8)
        ax.set_xlabel("USD per run")
        ax.set_ylabel("score")
        ax.set_title(f"Score against cost, one point per run: {label}")
        ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    nothing_to_draw("quality against cost")

## Performance against context length

Each point is one run: how many tokens the prompt of its last model call held (how full the context
was when the answer was produced) against its score, condition by colour and marker, on a log axis.
The lines are each condition's mean score in six log-spaced bands of context across the panel, with
a 95% bootstrap band where a band holds two runs or more. One panel per subject and model. Hollow
points come from records made before the calls were kept: their context is the run's prompt tokens
over its turns, a mean per call that understates the last call. It does not show why a run's context
grew, and a subagent's own context is not in it.

In [ ]:
fig = figures.context_curve(curve_runs)
if fig is None:
    nothing_to_draw("score against context")
else:
    plt.show()

## Best and mean score per condition

The solid bar is each condition's mean score over its runs with a 95% bootstrap interval; the hatched
bar takes the best repetition of each task and averages those over tasks, what the condition delivers
when it goes well. A wide gap between the two is a condition that is often right and sometimes far
off. One panel per subject and model. It does not show what either cost.

In [ ]:
fig = figures.best_and_mean(curve_runs)
if fig is None:
    nothing_to_draw("best and mean scores")
else:
    plt.show()

## Time per query

Left, the runner's wall time divided by the run's model turns: what each round trip cost in time,
tool work included. Right, seconds from the start of a run to the start of its answer, where the
agent could tell. One row per subject and model. A condition that takes fewer, slower turns can be
quicker to answer; read the two together.

In [ ]:
fig = figures.time_per_query(curve_runs)
if fig is None:
    nothing_to_draw("time per query")
else:
    plt.show()

## Cost against context

Each point is one run: the prompt size of its last call against what the run cost, condition by
colour and marker, hollow where approximated. A run's cost is all its calls, subagents' included,
and depends on how much of the prompt the cache served, so two runs at one context can cost very
different amounts. One panel per subject and model.

In [ ]:
fig = figures.cost_against_context(curve_runs)
if fig is None:
    nothing_to_draw("cost against context")
else:
    plt.show()

## Model tier

Where more than one model ran on a subject, the mean score by condition with one line per model and
its 95% interval, so a small model given the notes can be read against a large model given nothing.
Each line is drawn from that model's own runs; batches run at different hours share nothing else.
With one model per subject there is nothing to compare, and the cell says so.

In [ ]:
fig = figures.model_tier(curve_runs)
if fig is None:
    print("No subject has runs from more than one model yet.")
else:
    plt.show()

## Better, worse, mixed or tied

README.md § "What counts as better", applied per subject, agent and task to every pair of
conditions,
from the first condition's side. Means are over reps; a difference within the larger of the two
standard deviations, or from fewer than 5 runs on either side, is a tie. Better on cost needs a
score not lower and a cost lower; better on time, a score not lower and a time lower, where time
is the runner's wall time. Better on both is **better**; better on one and worse on the other is
**mixed**; a lower cost or time bought with a lower score, or a higher score bought with a higher
cost or time, is a **trade**. Non-ok runs score 0. `best` is each side's best repetition on the task, shown
beside the means and never deciding the verdict.

In [ ]:
def compare(a, b, column):
    a, b = a[column].dropna(), b[column].dropna()
    if len(a) < MIN_REPS or len(b) < MIN_REPS:
        return "tie"
    diff = a.mean() - b.mean()
    spread = max(a.std(ddof=1), b.std(ddof=1))
    if abs(diff) <= spread:
        return "tie"
    return "lower" if diff < 0 else "higher"


def verdict(a, b):
    score = compare(a, b, "score")
    sides = {m: compare(a, b, col) for m, col in (("cost", "cost_usd"), ("time", "wall_s"))}
    better = {m for m, s in sides.items() if s == "lower" and score in ("tie", "higher")}
    worse = {m for m, s in sides.items() if s == "higher" and score in ("tie", "lower")}
    trade = {m for m, s in sides.items() if (s == "lower" and score == "lower") or (s == "higher" and score == "higher")}
    if better == {"cost", "time"}:
        return "better"
    if worse == {"cost", "time"}:
        return "worse"
    if better and worse:
        return "mixed"
    if trade:
        return "trade"
    if better:
        return f"better on {next(iter(better))}"
    if worse:
        return f"worse on {next(iter(worse))}"
    return "tied"


rows = []
for (subject, agent, task), g in started.groupby(["subject", "agent", "task"]) if len(started) else []:
    present = [c for c in conditions if c in set(g["condition"])]
    for i, ca in enumerate(present):
        for cb in present[i + 1:]:
            a, b = g[g["condition"] == ca], g[g["condition"] == cb]
            rows.append({
                "subject": subject, "agent": agent, "task": task, "condition": ca, "against": cb,
                "runs": f"{len(a)} v {len(b)}",
                "score": f"{a['score'].mean():.2f} v {b['score'].mean():.2f}",
                "best": f"{a['score'].max():.2f} v {b['score'].max():.2f}",
                "cost": f"{a['cost_usd'].mean():.4f} v {b['cost_usd'].mean():.4f}",
                "wall s": f"{a['wall_s'].mean():.1f} v {b['wall_s'].mean():.1f}",
                "verdict": verdict(a, b),
            })
verdicts = pd.DataFrame(rows)
if len(verdicts):
    display(verdicts)
else:
    nothing_to_draw("a comparison")

## The configuration axis

Score and cost by the named Sloppy configuration a batch ran under (`--config`; `shipped` where
none was named), by condition, one row per subject and agent. With one configuration there is nothing to compare, and the cell
says so rather than drawing.

In [ ]:
configs = sorted(set(started["config"])) if len(started) else []
if len(configs) < 2:
    print(f"Runs under {len(configs)} configuration(s): {', '.join(configs) or 'none'}. Nothing to compare yet.")
else:
    fig, rows = panel_axes(started, ncols=2, width=11, height=3.8)
    width = 0.8 / len(configs)
    for axes, label, g in rows:
        here = ran_in(g)
        s = summary(g, ["config", "condition"], "score")
        c = summary(g, ["config", "condition"], "cost_usd")
        for ax, frame, ylabel in ((axes[0], s, "score"), (axes[1], c, "USD per run")):
            for i, cfg in enumerate(configs):
                part = frame[frame["config"] == cfg].set_index("condition").reindex(here)
                ax.bar(np.arange(len(here)) + i * width, part["mean"], width, color=PALETTE[i % len(PALETTE)], label=cfg)
            ax.set_xticks(np.arange(len(here)) + 0.4 - width / 2, here, rotation=20)
            ax.set_ylabel(ylabel)
            ax.set_title(label)
    rows[0][0][0].legend(frameon=False, title="configuration")
    fig.tight_layout()
    plt.show()